In [1]:
# ====================================================
# Libraries
# ====================================================

import numpy as np
import matplotlib.pyplot as plt
from getdist import MCSamples, plots

In [2]:
# ====================================================
# File paths
# ====================================================
# DMbudget+CCsubset robustness test: only the "with FRB" configuration is relevant
# here (the Galactic DM budget only enters through the FRB likelihood),
# so we compare the fiducial CC(35)+SNe+BAO+FRB run against the combined
# DMbudget+CCsubset robustness-test run: DM_MW,halo and DM_MW,ISM uncertainties
# marginalized/propagated (DMbudget), AND the two CC points correlated with the
# LEGA-C sample (Jimenez_2023, Jiao_2023) removed, leaving CC(33) (CCsubset).

fname_fid   = "/home/brunowesley/projetos/MCMC-cosmo/Codes/MCMC-Models/GR-based/LCDM/CC_SNe_BAO_FRB/flat_samples_LCDM_cc_sne_bao_frb.npy"
fname_robust = "/home/brunowesley/projetos/MCMC-cosmo/Codes/Robustness-test/LCDM/CC_SNe_BAO_FRB/flat_samples_LCDM_cc_sne_bao_frb_DMbudget_CCsubset.npy"


# ====================================================
# Load samples (FULL chains)
# ====================================================

samples_fid_full   = np.load(fname_fid)
samples_robust_full = np.load(fname_robust)

print("Fiducial (CC(35)+SNe+BAO+FRB):           ", samples_fid_full.shape)
print("DMbudget+CCsubset robustness test (CC(33)+SNe+BAO+FRB):", samples_robust_full.shape)

Fiducial (CC(35)+SNe+BAO+FRB):            (1909728, 6)
DMbudget+CCsubset robustness test (CC(33)+SNe+BAO+FRB): (278400, 6)


In [3]:
# ====================================================
# Function: marginalized statistics (PRINT + SAVE)
# ====================================================

def print_stats(flat_samples, param_names_indices, fmt_map, title, file=None):

    lines = []
    lines.append("\n" + "="*60)
    lines.append(f"{title:^60}")
    lines.append("="*60)

    for name, idx in param_names_indices:
        samples = flat_samples[:, idx]

        median  = np.percentile(samples, 50)
        lower68 = np.percentile(samples, 16)
        upper68 = np.percentile(samples, 84)
        lower95 = np.percentile(samples, 2.5)
        upper95 = np.percentile(samples, 97.5)

        ep68 = upper68 - median
        em68 = median  - lower68
        ep95 = upper95 - median
        em95 = median  - lower95

        fmt = fmt_map.get(name, ".3f")

        lines.append(f"\n  {name}:")
        lines.append(f"    {median:{fmt}} +{ep68:{fmt}} -{em68:{fmt}}  (68% CL)")
        lines.append(f"    {median:{fmt}} +{ep95:{fmt}} -{em95:{fmt}}  (95% CL)")

    lines.append("\n" + "="*60)

    # ---- Print on screen ----
    for line in lines:
        print(line)

    # ---- Save to file ----
    if file is not None:
        for line in lines:
            file.write(line + "\n")


# ====================================================
# Save ALL output to file
# ====================================================
# DMbudget+CCsubset robustness test: both chains use the same 6-parameter FRB model
# (H0, Om0, Ob, MB, sigma_host, exp_mu), so a single param/format map
# covers both -- no separate "no FRB" branch needed here.

output_file = "stats_LCDM_DMbudget_CCsubset.txt"

param_FRB = [
    ("H0", 0),
    ("Om0", 1),
    ("Ob", 2),
    ("MB", 3),
    ("sigma_host", 4),
    ("exp_mu", 5)
]

fmt_FRB = {
    "H0": ".2f",
    "Om0": ".3f",
    "Ob": ".4f",
    "MB": ".3f",
    "sigma_host": ".3f",
    "exp_mu": ".1f"
}

with open(output_file, "w", encoding="utf-8") as f:

    # ====================================================
    # Statistics: Fiducial (DM_MW,halo = 50 pc/cm^3 fixed,
    # DM_MW,ISM point estimate, no propagated systematic uncertainty;
    # CC(35), all measurements treated as independent)
    # ====================================================

    print_stats(
        samples_fid_full,
        param_FRB,
        fmt_FRB,
        "LCDM | CC(35) + SNe + BAO + FRB (Fiducial)",
        file=f
    )

    # ====================================================
    # Statistics: DMbudget+CCsubset robustness test (DM_MW,halo marginalized via
    # P_halo, DM_MW,ISM uncertainty sigma_MW added in quadrature [DMbudget];
    # CC(33), Jimenez_2023 and Jiao_2023 removed as correlated with the
    # Borghi_2022/LEGA-C measurement [CCsubset])
    # ====================================================

    print_stats(
        samples_robust_full,
        param_FRB,
        fmt_FRB,
        "LCDM | CC(33) + SNe + BAO + FRB (DMbudget+CCsubset robustness test)",
        file=f
    )

print(f"\n✅ Full output saved to: {output_file}")


         LCDM | CC(35) + SNe + BAO + FRB (Fiducial)         

  H0:
    70.46 +1.04 -1.06  (68% CL)
    70.46 +2.03 -2.13  (95% CL)

  Om0:
    0.305 +0.008 -0.008  (68% CL)
    0.305 +0.015 -0.015  (95% CL)

  Ob:
    0.0501 +0.0017 -0.0017  (68% CL)
    0.0501 +0.0033 -0.0034  (95% CL)

  MB:
    -19.338 +0.032 -0.033  (68% CL)
    -19.338 +0.061 -0.066  (95% CL)

  sigma_host:
    0.833 +0.124 -0.102  (68% CL)
    0.833 +0.269 -0.185  (95% CL)

  exp_mu:
    104.2 +16.2 -15.8  (68% CL)
    104.2 +32.6 -31.0  (95% CL)


  LCDM | CC(33) + SNe + BAO + FRB (DMbudget+CCsubset robustness test)  

  H0:
    70.51 +1.05 -1.09  (68% CL)
    70.51 +2.08 -2.19  (95% CL)

  Om0:
    0.305 +0.008 -0.008  (68% CL)
    0.305 +0.016 -0.015  (95% CL)

  Ob:
    0.0502 +0.0017 -0.0017  (68% CL)
    0.0502 +0.0034 -0.0034  (95% CL)

  MB:
    -19.336 +0.032 -0.034  (68% CL)
    -19.336 +0.063 -0.069  (95% CL)

  sigma_host:
    0.799 +0.121 -0.109  (68% CL)
    0.799 +0.243 -0.199  (95% CL)

  exp_mu

In [4]:
# ====================================================
# Parameters for corner plot
# ====================================================
# Both chains share the same 6-parameter FRB model, so no column
# subsetting is needed here (unlike the fiducial no-FRB vs FRB
# comparison, which had to restrict to the 4 shared parameters).

param_names  = ["H0", "Om0", "Ob", "MB", "sigma_host", "exp_mu"]
param_labels = [r"H_0", r"\Omega_{\rm m}", r"\Omega_{\rm b}",
                r"M_B", r"\sigma_{\rm host}", r"e^\mu"]

# ====================================================
# GetDist
# ====================================================

gd_fid   = MCSamples(samples=samples_fid_full,
                     names=param_names,
                     labels=param_labels)

gd_robust = MCSamples(samples=samples_robust_full,
                     names=param_names,
                     labels=param_labels)

for gd in [gd_fid, gd_robust]:
    gd.updateSettings({
        "smooth_scale_1D": 0.25,
        "smooth_scale_2D": 0.25,
        "fine_bins": 1024,
        "fine_bins_2D": 1024,
    })

# ====================================================
# Corner plot
# ====================================================

g = plots.get_subplot_plotter()

g.triangle_plot(
    [gd_fid, gd_robust],
    filled=True,
    legend_labels=[r"Fiducial", r"$\mathrm{DM_{MW}}$ systematics + CC(33)"],
    contour_colors=["red", "navy"],
    title_limit=0
)

plt.savefig("Corner_LCDM_DMbudget_CCsubset.png", dpi=300, bbox_inches="tight")
plt.savefig("Corner_LCDM_DMbudget_CCsubset.pdf", dpi=300, bbox_inches="tight")
plt.show()

Removed no burn in
Removed no burn in


In [5]:
# ========================================================
# Derived Parameter: Sound Horizon rd
# ========================================================

Neff = 3.046 

def compute_rd_chain(flat_samples):
    H0  = flat_samples[:, 0]
    Om0 = flat_samples[:, 1]
    Ob  = flat_samples[:, 2]

    h2   = (H0 / 100.0)**2
    Obh2 = Ob * h2
    Omh2 = Om0 * h2

    rd = 147.05 * ((Obh2 / 0.02236)**-0.13) \
                 * ((Omh2 / 0.1432)**-0.23) \
                 * ((Neff  / 3.04)**-0.1)

    return rd


def print_rd_stats(rd_chain, title):

    median  = np.percentile(rd_chain, 50)
    lower68 = np.percentile(rd_chain, 16)
    upper68 = np.percentile(rd_chain, 84)
    lower95 = np.percentile(rd_chain, 2.5)
    upper95 = np.percentile(rd_chain, 97.5)

    ep68 = upper68 - median
    em68 = median  - lower68
    ep95 = upper95 - median
    em95 = median  - lower95

    print("\n" + "="*60)
    print(f"{title:^60}")
    print("="*60)

    print(f"\n  r_d:")
    print(f"    {median:.3f} +{ep68:.3f} -{em68:.3f}  (68% CL)")
    print(f"    {median:.3f} +{ep95:.3f} -{em95:.3f}  (95% CL)")

    print("\n" + "="*60)


# ========================================================
# Compute rd for both chains
# ========================================================

rd_fid   = compute_rd_chain(samples_fid_full)
rd_robust = compute_rd_chain(samples_robust_full)


# ========================================================
# Print results
# ========================================================

print_rd_stats(rd_fid,   "Sound Horizon r_d | Fiducial (CC(35) + SNe + BAO + FRB)")
print_rd_stats(rd_robust, "Sound Horizon r_d | DMbudget+CCsubset robustness test (CC(33) + SNe + BAO + FRB)")


  Sound Horizon r_d | Fiducial (CC(35) + SNe + BAO + FRB)   

  r_d:
    143.137 +2.185 -2.067  (68% CL)
    143.137 +4.436 -3.990  (95% CL)


Sound Horizon r_d | DMbudget+CCsubset robustness test (CC(33) + SNe + BAO + FRB)

  r_d:
    143.035 +2.241 -2.123  (68% CL)
    143.035 +4.572 -4.135  (95% CL)

